In [10]:
import datetime, pytz, time, httpx
from urllib.parse import urlencode
import os, asyncio, httpx, pyperclip, arrow, requests, time, hmac, hashlib
import pandas as pd
import pytz
from binance import AsyncClient, BinanceSocketManager
from dotenv import load_dotenv
from nbclient.client import timestamp

from exchange.Trades import Order


load_dotenv('../../.env')
BINANCE_KEY = os.getenv('BINANCE_KEY')
BINANCE_SECRET = os.getenv('BINANCE_SECRET')

client = await AsyncClient.create(BINANCE_KEY, BINANCE_SECRET)

## Account

In [30]:
# status = await client.get_account_api_trading_status()
data = await client.get_trade_fee()
feedf = pd.DataFrame(data)
feedf[feedf['symbol'] == 'zrousdt'.upper()]

,symbol,makerCommission,takerCommission
3035,ZROUSDT,0.001,0.001


In [144]:
start_time = arrow.get('2025-04').int_timestamp * 1000
timestamp = int(time.time() * 1000)
tradesdf = pd.DataFrame(await client.get_my_trades(symbol='zrousdt'.upper(), startTime=start_time, timestamp=timestamp))
# tradesdf[['symbol', 'quoteQty', 'commission']]
# tradesdf['time'] = pd.to_datetime(tradesdf['time'], unit='ms')
# tradesdf['time']
tradesdf

""


In [34]:
assetdf = pd.DataFrame(await client.get_asset_details()).T
assetdf

,withdrawFee,minWithdrawAmount,withdrawStatus,depositStatus,depositTip
AGLD,2.21,4.42,True,True,NaN
STPT,43,86,True,True,NaN
SCR,0.2,4,True,True,NaN
ATEM,2.1,4.2,True,True,NaN
UGX,0,0,True,True,NaN
...,...,...,...,...,...
1000CHEEMS,355,710,True,True,NaN
NZD,0,0,True,True,NaN
AKRO,3091,6182,True,False,"Wallet Maintenance, Deposit Suspended"
1000PEPPER,75000,150000,True,False,NaN


## Orders

In [34]:
async def clean_orders(df_: pd.DataFrame):
    df_['time'] = pd.to_datetime(df_['time'], unit='ms')
    df_['updateTime'] = pd.to_datetime(df_['updateTime'], unit='ms')
    df_['workingTime'] = pd.to_datetime(df_['workingTime'], unit='ms')
    df_ = df_.drop(columns=['workingTime', 'selfTradePreventionMode', 'isWorking', 'orderListId'])
    df_ = df_.rename(columns={
        'clientOrderId': 'client_orderid',
        'origQty': 'amount',
        'executedQty': 'executed_amount',
        'cummulativeQuoteQty': 'cum_quote_amount',
        'timeInForce': 'time_in_force',
        'stopPrice': 'stop_price',
        'icebergQty': 'iceberg_amount',
        'origQuoteOrderQty': 'quote_amount',
        'time': 'created_at',
        'updateTime': 'updated_at',
    })
    # df_[df_['price'] == '0.00000000']
    return df_


def df_to_sql_insert(df, table_name="orders"):
    columns = ','.join(df.columns)
    insert_template = f"INSERT INTO {table_name} ({columns}) VALUES "
    values = [
        f"({','.join([repr(val) for val in row])})"
        for row in df.itertuples(index=False)
    ]
    return insert_template + ',\n'.join(values) + ";"

In [59]:
# basedf = pd.DataFrame(await client.get_all_orders(symbol='BANANAUSDT')).set_index('orderId')
# ordersdf = await clean_orders(basedf.copy())
# # ordersdf.loc[386446134]
# ordersdf

start_time = arrow.get('2025-04-01').int_timestamp
ordersdf = pd.DataFrame(await client.get_all_orders(symbol='BANANAUSDT', startTime=1743465600)).set_index('orderId')
ordersdf['time'] = pd.to_datetime(ordersdf['time'], unit='ms')
ordersdf['updateTime'] = pd.to_datetime(ordersdf['updateTime'], unit='ms')
ordersdf[['time', 'updateTime']]
# start_time

,time,updateTime
orderId,,
376714213,2025-03-25 22:39:45.828,2025-03-25 22:41:31.773
376715538,2025-03-25 22:43:03.776,2025-03-26 00:42:57.982
383577752,2025-03-29 22:38:17.693,2025-03-29 22:47:48.977
383583165,2025-03-29 22:49:37.490,2025-03-30 02:34:30.003
385499582,2025-03-31 17:18:08.710,2025-04-01 05:18:06.711
386446134,2025-04-01 05:18:10.502,2025-04-01 05:18:10.502
386831406,2025-04-01 10:20:40.155,2025-04-01 10:20:40.155
387415847,2025-04-01 16:21:50.289,2025-04-04 16:28:39.915
392834280,2025-04-04 16:29:49.769,2025-04-04 16:29:49.769


## Request using HMAC

In [18]:
# API credentials
BASE_URL = "https://api.binance.com"
api_endpoint = f'{BASE_URL}/api/v3/allOrders'

# Parameters
symbol = "REDUSDT"
start_time = arrow.get("2025-04").int_timestamp * 1000
timestamp = int(time.time() * 1000)  # Current time to prevent replay attacks
params = {
    "symbol": symbol,
    "startTime": start_time,
    "timestamp": timestamp,
}
query_string = urlencode(params)
signature = hmac.new(bytes(BINANCE_SECRET, 'utf-8'), bytes(query_string, 'utf-8'), hashlib.sha256).hexdigest()
params["signature"] = signature

# # Demo for verifying the data
# params = {
#     "symbol": symbol,
#     "startTime": start_time,
#     "timestamp": timestamp,
# }
# query_string = urlencode(params)
# signature2 = hmac.new(bytes(BINANCE_SECRET, 'utf-8'), bytes(query_string, 'utf-8'), hashlib.sha256).hexdigest()
# is_valid = hmac.compare_digest(signature, signature2)
# is_valid

# Send request
headers = {"X-MBX-APIKEY": BINANCE_KEY}
# response = requests.get(
#     f"{BASE_URL}/api/v3/allOrders",
#     headers=headers,
#     params=params,
# )
# ordersdf = pd.DataFrame(response.json())
response = httpx.get(api_endpoint, headers=headers, params=params)
ordersdf = pd.DataFrame(response.json())
ordersdf['time'] = pd.to_datetime(ordersdf['time'], unit='ms')
ordersdf['updateTime'] = pd.to_datetime(ordersdf['updateTime'], unit='ms')
ordersdf[['symbol', 'time', 'updateTime']]


,symbol,time,updateTime
0,REDUSDT,2025-04-01 06:27:35.055,2025-04-01 06:27:35.055


## Request using Public Key

In [48]:
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from os.path import expanduser
import base64, time
from cryptography.hazmat.primitives.serialization import load_pem_private_key, load_ssh_private_key


# Generate keys
# !openssl genpkey -algorithm ED25519 -out private.pem
# !openssl pkey -in private.pem -pubout -out public.pem

# # Load Ed25519 private key (PKCS#8 format)
# with open(expanduser('~/Desktop/Downloads/aaa/private.pem'), 'rb') as f:
#     private_key = Ed25519PrivateKey.from_private_bytes(
#         serialization.load_pem_private_key(f.read(), password=None).private_bytes(
#             encoding=serialization.Encoding.Raw,
#             format=serialization.PrivateFormat.Raw,
#             encryption_algorithm=serialization.NoEncryption()
#         )
#     )

# with open(expanduser('~/Desktop/Downloads/aaa/priv2'), 'rb') as f:
#     private_key = serialization.load_ssh_private_key(f.read(), password=None)

with open(expanduser('~/Desktop/Downloads/aaa/private.pem'), 'rb') as f:
    private_key = load_pem_private_key(data=f.read(), password=None)


# API params
params = {
    "symbol": 'REDUSDT',
    "startTime": arrow.get("2025-04").int_timestamp * 1000,
    'timestamp': int(time.time() * 1000),  # Current time to prevent replay attacks
}
query_string = urlencode(params)

# Sign with Ed25519
signature = private_key.sign(query_string.encode('utf-8')).hex()
# signature = base64.b64encode(private_key.sign(query_string.encode('ASCII')))
params["signature"] = signature

# Send the request
headers = {
    'X-MBX-APIKEY': BINANCE_KEY,
}

# with open(expanduser('~/Desktop/Downloads/aaa/binance.enchance_ed25519'), 'rb') as f:
#     private_key = load_pem_private_key(data=f.read(), password=None)
#
# # Sign the request
# # payload = '&'.join([f'{param}={value}' for param, value in params.items()])
# payload = urlencode(params)
# signature = base64.b64encode(private_key.sign(payload.encode('ASCII')))
# params['signature'] = signature

base_url = "https://api.binance.com"
api_endpoint = f'{base_url}/api/v3/allOrders'
response = httpx.get(
    api_endpoint,
    headers=headers,
    params=params,
)
response

<Response [400 Bad Request]>

In [42]:
# sql = df_to_sql_insert(ordersdf, 'orders')
# sql
ll = []
# for idx, row in zip(ordersdf.index, ordersdf.to_numpy()):
for row in ordersdf.itertuples(index=True):
    dd = row._asdict()
    dd['id'] = dd['Index']
    del dd['Index']
    order = Order(**dd)
    print(order)
    # dict_ = {
    #     'id': idx,
    #     'symbol': row[0],
    #     'price': row[3],
    #     'amount': row[4],
    #     'executed_amount': row[5],
    #     'cumulative_amount': row[6],
    #     'status': row[7],
    #     'time_in_force': row[8],
    #     'type': row[9],
    #     'side': row[10],
    #     'stop_price': row[11],
    #     'iceberg': row[12],
    #     'implemented_at': row[13],
    #     'quote_amount': row[15],
    #     # 'exchange_id': 1,
    # }
    # print(dict_)

InvalidRequestError: When initializing mapper Mapper[Taxonomy(app_taxonomy)], expression 'Account' failed to locate a name ('Account'). If this is a class name, consider adding this relationship() to the <class 'models.common_models.Taxonomy'> class after both dependent classes have been defined.

## Trades

In [2]:


# async def trade_history():
#     # bsm = BinanceSocketManager(client)

account_trades = await client.get_account()
# traded_symbols = {balance['asset'] + "USDT" for balance in account_trades['balances']}  # Adjust for different pairs
# traded_symbols = {bal['asset']: bal for bal in account_trades['balances'] if float(bal['free'])}
# traded_symbols = [bal for bal in account_trades['balances'] if float(bal['free'])]

# Fetch trades for each symbol
# tasks = [client.get_my_trades(symbol=symbol) for symbol in traded_symbols]
# all_trades = await asyncio.gather(*tasks, return_exceptions=True)
# ic(all_trades[0])

trade_history = [bal for bal in account_trades['balances'] if float(bal['free']) > 0 or float(bal['locked']) > 0]

df = pd.DataFrame(trade_history)  # noqa
df

NameError: name 'client' is not defined

In [190]:
df = df[(df['free'].astype(float) >= 1) | (df['locked'].astype(float) >= 1)]
# df['asset'].unique()

In [234]:
tasks = [client.get_my_trades(symbol=f'{symbol}USDT') for symbol in df['asset']]
trades = await asyncio.gather(*tasks, return_exceptions=True)
tradesdf = pd.concat([pd.DataFrame(trade) for trade in trades], ignore_index=True)
tradesdf['time'] = pd.to_datetime(tradesdf['time'], unit='ms')
tradesdf = tradesdf.set_index('id').sort_values(by='time').sort_values(by='id')
tradesdf

,symbol,orderId,orderListId,price,qty,quoteQty,commission,commissionAsset,time,isBuyer,isMaker,isBestMatch
id,,,,,,,,,,,,
2710807,REDUSDT,36848173,-1,0.53980000,172.10000000,92.89958000,0.17210000,RED,2025-03-14 16:21:05.713,True,False,True
2710808,REDUSDT,36848173,-1,0.53980000,66.30000000,35.78874000,0.06630000,RED,2025-03-14 16:21:05.713,True,False,True
2710809,REDUSDT,36848173,-1,0.53980000,13.90000000,7.50322000,0.01390000,RED,2025-03-14 16:21:05.713,True,False,True
2710810,REDUSDT,36848173,-1,0.54010000,4.20000000,2.26842000,0.00420000,RED,2025-03-14 16:21:05.713,True,False,True
2710811,REDUSDT,36848173,-1,0.54010000,21.30000000,11.50413000,0.02130000,RED,2025-03-14 16:21:05.713,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
39390706,ZROUSDT,845588788,-1,3.22300000,103.50000000,333.58050000,0.10350000,ZRO,2025-03-27 06:59:30.675,True,False,True
39390707,ZROUSDT,845588788,-1,3.22300000,99.81000000,321.68763000,0.09981000,ZRO,2025-03-27 06:59:30.675,True,False,True
118591245,EGLDUSDT,1910149777,-1,18.85000000,9.22000000,173.79700000,0.00922000,EGLD,2025-03-27 12:32:15.535,True,False,True


In [237]:
# tradesdf['symbol'].unique()
tradesdf.columns

Index(['symbol', 'orderId', 'orderListId', 'price', 'qty', 'quoteQty',
       'commission', 'commissionAsset', 'time', 'isBuyer', 'isMaker',
       'isBestMatch'],
      dtype='object')

In [ ]:
tradesdf[tradesdf['isBuyer']].sample(10)
tradesdf[(tradesdf['isBuyer']) & (tradesdf['symbol'] == 'BANANAUSDT') & (tradesdf['orderId'] == 383583165)]

In [271]:
comm = 0.001
total = 19.41000000 * 11.87900000
# fee = total * comm
expense = total + 0.01187900
# fee
# total
expense

230.58326899999997

In [238]:
tradesdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 116 entries, 2710807 to 118591247
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   symbol           116 non-null    object        
 1   orderId          116 non-null    int64         
 2   orderListId      116 non-null    int64         
 3   price            116 non-null    object        
 4   qty              116 non-null    object        
 5   quoteQty         116 non-null    object        
 6   commission       116 non-null    object        
 7   commissionAsset  116 non-null    object        
 8   time             116 non-null    datetime64[ns]
 9   isBuyer          116 non-null    bool          
 10  isMaker          116 non-null    bool          
 11  isBestMatch      116 non-null    bool          
dtypes: bool(3), datetime64[ns](1), int64(2), object(6)
memory usage: 9.4+ KB
